In [0]:
import requests
import time
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("entity_name", "ticks")
dbutils.widgets.text("load_pattern", "1")
dbutils.widgets.text("SUBGRAPH_URL", "https://gateway.thegraph.com/api/a5bbc98aac75dac555f9aba8747c7e2e/subgraphs/id/5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV")

In [0]:
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

In [0]:
PAGE_SIZE = 1000
ticks_list = []
query_variables= {
    "pool_id": "",
    "skip": 0
}

In [0]:
ticks_query = """
query TicksQuery($pool_id: String!, $skip: Int!) {
  ticks(
    where: {
      pool: $pool_id
    }
    first: 1000,
    skip: $skip, 
  ){
    id
    poolAddress
    tickIdx
    pool
    volumeToken0
    volumeToken1
    volumeUSD
    price0
    price1
    liquidityGross
    liquidityNet
    feesUSD
    collectedFeesToken0
    collectedFeesToken1
    collectedFeesUSD
    createdAtTimestamp
    createdAtBlockNumber
  }
}
"""

In [0]:
pools_df = spark.read.table("uniswap.bronze.pools").filter("liquidity != '0'")

In [0]:
display(pools_df)

In [0]:
for row in pools_df.collect():
    print(f"*INFO*: Processing ticks from pool: {row.token0["symbol"]}/{row.token1["symbol"]} | {row.id}.")
    query_variables.update({"pool_id": row.id})

    while True:
        print(f"*INFO*: Skip rows amount: {query_variables["skip"]}.")
        response = requests.post(SUBGRAPH_URL, json={"query": ticks_query, "variables": query_variables})
        ticks_list.extend(response.json()["data"][entity_name])

        time.sleep(0.3)

        rows_loaded = len(response.json()["data"][entity_name])
        print(f"*INFO*: Number of rows loaded: {rows_loaded}.")

        if rows_loaded == PAGE_SIZE:
            query_variables.update({"skip": query_variables["skip"] + PAGE_SIZE}) 
        elif rows_loaded == 0: 
            break
        else:
            query_variables.update({"skip": query_variables["skip"] + rows_loaded})  

    query_variables.update({"skip": 0}) 

In [0]:
ticks_df = spark.createDataFrame(ticks_list)
ticks_df = ticks_df.withColumn("load_timestamp", current_timestamp())

In [0]:
display(ticks_df)

In [0]:
if load_pattern == "1":
    print("*INFO*: Executing full load.")
    ticks_df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "2":
    print("*INFO*: Executing incremental load.")
    ticks_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "3":
    print("*INFO*: Executing differential load.")
    old_ticks_table = DeltaTable.forName(spark, f"uniswap.bronze.{entity_name}")
    old_ticks_table.alias("old").merge(
        ticks_df.alias("new"),
        # "old.id = new.id AND old.token0.id = new.token0.id AND old.token1.id = new.token1.id"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")
else:
    print("*ERROR*: Invalid load pattern.")
    dbutils.notebook.exit("Invalid load pattern.")